# 第 4 章 复购预测与触达名单

本章商业问题：**预算有限时，应该优先触达哪些用户，阈值怎么选才赚钱？**

本章所有代码默认优先读取真实 ETL 接口 `http://192.168.31.47:38173/api/etl`。如果接口暂时不可用，会自动回退到本地 SQLite 后备数据，保证课堂可以继续进行。

## 0. 先建立商业问题意识

在商业课里，代码不是第一步。第一步是判断：这个问题影响收入、成本、用户体验、库存风险，还是营销效率？只有先说清楚商业目标，后面的数据选择和模型选择才不会跑偏。

In [ ]:
from pathlib import Path
import sys

COURSE_ROOT = Path.cwd()
if COURSE_ROOT.name in ["notebooks", "student_notebooks", "teacher_notebooks"]:
    COURSE_ROOT = COURSE_ROOT.parent
elif not (COURSE_ROOT / "course_utils").exists():
    COURSE_ROOT = Path("..").resolve()

sys.path.insert(0, str(COURSE_ROOT))

from course_utils.data_loader import (
    API_BASE, load_table, get_metrics, get_quality_report,
    get_table_catalog, get_schema, paid_orders, api_status, query_table
)
from course_utils.business import money, pct, section

try:
    import matplotlib.pyplot as plt
    plt.rcParams["font.sans-serif"] = ["Microsoft YaHei", "SimHei", "Arial Unicode MS", "DejaVu Sans"]
    plt.rcParams["axes.unicode_minus"] = False
except Exception:
    pass

print("课程目录:", COURSE_ROOT)
print("ETL API:", API_BASE)
print("API 状态:", api_status())

## 1. 查看 ETL 数据资产

下面先查看 ETL 接口暴露了哪些表。请注意 `dim_` 开头的是维度表，通常描述对象；`fact_` 开头的是事实表，通常记录业务事件。

In [ ]:
catalog = get_table_catalog()
tables = catalog["tables"]
print("可用表数量:", catalog.get("total", len(tables)))
for t in tables[:12]:
    print(t["tableName"], t.get("recordCount"), t.get("type"), t.get("description", ""))

## 2. 定义复购标签

标签定义决定模型训练目标。这里用 2026-01-01 之前作为观察窗口，之后 60 天是否下单作为复购标签。

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, precision_score, recall_score

orders = paid_orders()
cutoff = pd.Timestamp("2026-01-01")
history = orders[orders["order_date"] < cutoff]
future = orders[(orders["order_date"] >= cutoff) & (orders["order_date"] < cutoff + pd.Timedelta(days=60))]

feat = history.groupby("user_id").agg(
    order_count=("order_id", "nunique"),
    total_paid=("paid_amount", "sum"),
    avg_paid=("paid_amount", "mean"),
    last_order_date=("order_date", "max")
).reset_index()
feat["recency_days"] = (cutoff - feat["last_order_date"]).dt.days
feat["label_repurchase"] = feat["user_id"].isin(future["user_id"].unique()).astype(int)
feat = feat.drop(columns=["last_order_date"]).fillna(0)
feat["label_repurchase"].mean()

In [ ]:
X = feat[["order_count", "total_paid", "avg_paid", "recency_days"]]
y = feat["label_repurchase"]
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
model = RandomForestClassifier(n_estimators=120, random_state=42, n_jobs=-1, min_samples_leaf=5)
model.fit(X_train, y_train)
proba = model.predict_proba(X_val)[:, 1]
auc = roc_auc_score(y_val, proba)
print("AUC:", round(auc, 3))

In [ ]:
rows = []
for threshold in np.arange(0.1, 0.9, 0.1):
    pred = (proba >= threshold).astype(int)
    touched = int(pred.sum())
    if touched == 0:
        continue
    precision = precision_score(y_val, pred, zero_division=0)
    recall = recall_score(y_val, pred, zero_division=0)
    expected_margin = touched * precision * 80
    touch_cost = touched * 8
    roi = (expected_margin - touch_cost) / max(touch_cost, 1)
    rows.append([threshold, touched, precision, recall, roi])
roi_table = pd.DataFrame(rows, columns=["threshold", "touch_users", "precision", "recall", "expected_roi"])
roi_table.sort_values("expected_roi", ascending=False)

In [ ]:
assert auc >= 0.5
print("第 04 章验证通过")

## 学生作业

请补充 150 到 300 字商业解读，至少引用两个数据结果，并给出一个明确运营动作。